<a href="https://colab.research.google.com/github/deepakri201/SR_for_NLST_Sybil/blob/main/demo/NLST_Sybil_FM_demo_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLST_Sybil_FM_demo_part1

In this notebook, we use metadata extracted from the DICOM SR files (present in BQ tables), and join with the clinical metadata already in IDC.

Deepa Krishnaswamy

Brigham and Women's Hospital

August 2025

Notes:
- Colab Pro
- Tables that hold the DICOM SR metadata: idc-external-018.sr_nlst_sybil.bbox_measurements

In [8]:
# SET THESE OPTIONS

# Create the BQ tables to hold the measurements from the SRs
# If not, load from csv file in github
create_bq_tables = 1

# Parameterization

In [1]:
#@title Enter your Project ID here
# initialize this variable with your Google Cloud Project ID!
project_name = "idc-external-018" #@param {type:"string"}

import os
os.environ["GCP_PROJECT_ID"] = project_name

!gcloud config set project $project_name

from google.colab import auth
auth.authenticate_user()

Updated property [core/project].


# Environment Setup

In [2]:
!pip install idc-index

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 52.4 MB/s eta 0:00:00
  Attempting uninstall: duckdb
    Found existing installation: duckdb 1.3.2
    Uninstalling duckdb-1.3.2:
      Successfully uninstalled duckdb-1.3.2


In [3]:
import os
import sys
import time

import numpy as np
import pandas as pd
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

import json
from pathlib import Path

In [4]:
from google.cloud import bigquery
from google.cloud import storage

In [5]:
from idc_index import IDCClient

idc_client = IDCClient.client()

In [15]:
# Download the BQ queries

!wget -O /content/measurement_groups.sql https://raw.githubusercontent.com/deepakri201/SR_for_NLST_Sybil/main/sql/measurement_groups.sql
!wget -O /content/bbox_measurements.sql https://raw.githubusercontent.com/deepakri201/SR_for_NLST_Sybil/main/sql/bbox_measurements.sql


--2025-08-05 21:04:28--  https://raw.githubusercontent.com/deepakri201/SR_for_NLST_Sybil/main/sql/measurement_groups.sql
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7156 (7.0K) [text/plain]
Saving to: ‘/content/measurement_groups.sql’

/content/measuremen 100%[===================>]   6.99K  --.-KB/s    in 0s      

2025-08-05 21:04:28 (62.6 MB/s) - ‘/content/measurement_groups.sql’ saved [7156/7156]

--2025-08-05 21:04:28--  https://raw.githubusercontent.com/deepakri201/SR_for_NLST_Sybil/main/sql/bbox_measurements.sql
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request se

In [6]:
# Download the measurements extracted from the DICOM SR files

!wget https://github.com/deepakri201/SR_for_NLST_Sybil/releases/download/v1.0.0/bbox_measurements.csv

--2025-08-05 20:43:51--  https://github.com/deepakri201/SR_for_NLST_Sybil/releases/download/v1.0.0/bbox_measurements.csv
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/983681157/58da2c36-a7d3-43b2-b57d-50cc0709c40f?sp=r&sv=2018-11-09&sr=b&spr=https&se=2025-08-05T21%3A35%3A28Z&rscd=attachment%3B+filename%3Dbbox_measurements.csv&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2025-08-05T20%3A34%3A51Z&ske=2025-08-05T21%3A35%3A28Z&sks=b&skv=2018-11-09&sig=sbaIzARg4EAYdDPqAh1eX2MjHiendewD0LU8OHqABA8%3D&jwt=eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc1NDQyNjkzMSwibmJmIjoxNzU0NDI2NjMxLCJwYXRoIjoicmVsZWFzZWFz

# Get the bounding box information

## df_sr - get the metadata from the SRs

In [16]:
query_measurement_groups_filename = "/content/measurement_groups.sql"
with open(query_measurement_groups_filename, 'r') as file:
  query_measurement_groups = file.read()
print(query_measurement_groups)

query_bbox_measurements_filename = "/content/bbox_measurements.sql"
with open(query_bbox_measurements_filename, 'r') as file:
  query_bbox_measurements = file.read()
print(query_bbox_measurements)


WITH
  measurementGroups AS (
  WITH
    contentSequenceLevel1 AS (
    WITH
      structuredReports AS (
      SELECT
        PatientID,
        SOPInstanceUID,
        SeriesInstanceUID,
        SeriesDescription,
        ContentSequence
      FROM
        `idc-external-018.sr_nlst_sybil.dicom_all`
      WHERE
        ( SOPClassUID = "1.2.840.10008.5.1.4.1.1.88.11"
          OR SOPClassUID = "1.2.840.10008.5.1.4.1.1.88.22"
          OR SOPClassUID = "1.2.840.10008.5.1.4.1.1.88.33"
          OR SOPClassUID = "1.2.840.10008.5.1.4.1.1.88.34"
          OR SOPClassUID = "1.2.840.10008.5.1.4.1.1.88.35" )
        AND ARRAY_LENGTH(ContentTemplateSequence) <> 0
        AND ContentTemplateSequence [
      OFFSET
        (0)].TemplateIdentifier = "1500"
        AND ContentTemplateSequence [
      OFFSET
        (0)].MappingResource = "DCMR" )
    SELECT
      PatientID,
      SOPInstanceUID,
      SeriesInstanceUID,
      SeriesDescription,
      contentSequence
    FROM
      structuredReports

In [18]:
### If create the BQ tables, run two queries and save the results ###
### Requires access to the original DICOM SR files and DICOM store ###
### Else load the measurements from a github release attachment ###

if create_bq_tables:

  # Create the table below using the query_measurement_groups
  # `idc-external-018.sr_nlst_sybil.measurement_groups`
  client_bq = bigquery.Client(project=project_name)
  destination_table_id_measurement_groups = "idc-external-018.sr_nlst_sybil.measurement_groups2"
  job_config = bigquery.QueryJobConfig(destination=destination_table_id_measurement_groups)
  job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
  query_job = client_bq.query(query_measurement_groups, job_config=job_config)
  query_job.result()
  print(f"Query results saved to table: {destination_table_id_measurement_groups}")

  # Create the table below using the query_bbox_measurements
  # `idc-external-018.sr_nlst_sybil.bbox_measurements`
  destination_table_id_bbox_measurements = "idc-external-018.sr_nlst_sybil.bbox_measurements2"
  job_config = bigquery.QueryJobConfig(destination=destination_table_id_bbox_measurements)
  job_config.write_disposition = bigquery.WriteDisposition.WRITE_TRUNCATE
  query_job = client_bq.query(query_bbox_measurements, job_config=job_config)
  query_job.result()
  print(f"Query results saved to table: {destination_table_id_bbox_measurements}")

  # Then query that table
  query = f"""
      SELECT
        *
      FROM
        `idc-external-018.sr_nlst_sybil.bbox_measurements`
        """
  df_sr = client_bq.query(query).to_dataframe()

else:

  df_sr = pd.read_csv("/content/bbox_measurements.csv")

Query results saved to table: idc-external-018.sr_nlst_sybil.measurement_groups2
Query results saved to table: idc-external-018.sr_nlst_sybil.bbox_measurements2


In [38]:
# Rename the column so we know it's the SeriesInstanceUID of the SR

df_sr = df_sr.rename(columns={'SeriesInstanceUID':"SR_SeriesInstanceUID"})

In [39]:
# First add columns for the width, height, center_x, and center_y

width_list = []
height_list = []
center_x_list = []
center_y_list = []

for index, row in df_sr.iterrows():
  # Get values
  x0 = row['x0']; y0 = row['y0']
  x1 = row['x1']; y1 = row['y1']
  x2 = row['x2']; y2 = row['y2']
  x3 = row['x3']; y3 = row['y3']
  # calculate the width, height and center, as these are needed for display
  min_x = np.min([x0, x1, x2, x3]) # using roi.GraphicData: min_x = np.min([bbox[0], bbox[2], bbox[4], bbox[6]])
  max_x = np.max([x0, x2, x2, x3]) # using roi.GraphicData: max_x = np.max([bbox[0], bbox[2], bbox[4], bbox[6]])
  min_y = np.min([y0, y1, y2, y3]) # using roi.GraphicData: min_y = np.min([bbox[1], bbox[3], bbox[5], bbox[7]])
  max_y = np.max([y0, y1, y2, y3]) # using roi.GraphicData: max_y = np.max([bbox[1], bbox[3], bbox[5], bbox[7]])
  width = max_x - min_x
  height = max_y - min_y
  center_x = min_x + width/2
  center_y = min_y + height/2
  # append
  width_list.append(width)
  height_list.append(height)
  center_x_list.append(center_x)
  center_y_list.append(center_y)

# Add columns
df_sr['width'] = width_list
df_sr['height'] = height_list
df_sr['center_x'] = center_x_list
df_sr['center_y'] = center_y_list

df_sr.head()

,PatientID,SR_SeriesInstanceUID,SOPInstanceUID,trackingIdentifier,trackingUniqueIdentifier,finding,findingSite,ReferencedSOPInstanceUID,ConceptNameCodeSequence,GraphicType,...,x1,y1,x2,y2,x3,y3,width,height,center_x,center_y
0,100012,1.2.826.0.1.3680043.8.498.37369100471267400346...,1.2.826.0.1.3680043.8.498.47967329963589463487...,1,1.2.826.0.1.3680043.8.498.51805212992957195930...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.33353059142567790698519649...,"[{'CodeValue': '111030', 'CodingSchemeDesignat...",POLYLINE,...,-30.834309,-144.752594,-30.834309,-124.700500,-53.438473,-124.700500,22.604164,20.052094,-42.136391,-134.726547
1,100012,1.2.826.0.1.3680043.8.498.37369100471267400346...,1.2.826.0.1.3680043.8.498.47967329963589463487...,2,1.2.826.0.1.3680043.8.498.80904594057742061722...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.33357115270405040538140383...,"[{'CodeValue': '111030', 'CodingSchemeDesignat...",POLYLINE,...,-30.914061,-144.023438,-30.914061,-123.789062,-53.335934,-123.789062,22.421873,20.234375,-42.124997,-133.906250
2,100012,1.2.826.0.1.3680043.8.498.37369100471267400346...,1.2.826.0.1.3680043.8.498.47967329963589463487...,3,1.2.826.0.1.3680043.8.498.13506178560260701077...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.29288147675470021330187068...,"[{'CodeValue': '111030', 'CodingSchemeDesignat...",POLYLINE,...,-30.914061,-142.929688,-30.914061,-122.695312,-53.882809,-122.695312,22.968748,20.234375,-42.398435,-132.812500
3,100012,1.2.826.0.1.3680043.8.498.37369100471267400346...,1.2.826.0.1.3680043.8.498.47967329963589463487...,4,1.2.826.0.1.3680043.8.498.77966303227192540697...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.26643655587266484144522325...,"[{'CodeValue': '111030', 'CodingSchemeDesignat...",POLYLINE,...,-30.914061,-142.382812,-30.914061,-121.601562,-53.882809,-121.601562,22.968748,20.781250,-42.398435,-131.992188
4,100012,1.2.826.0.1.3680043.8.498.37369100471267400346...,1.2.826.0.1.3680043.8.498.47967329963589463487...,5,1.2.826.0.1.3680043.8.498.60051261633302288013...,"{'CodeValue': '52988006', 'CodingSchemeDesigna...","{'CodeValue': '39607008', 'CodingSchemeDesigna...",1.2.840.113654.2.55.59559855445942972046043598...,"[{'CodeValue': '111030', 'CodingSchemeDesignat...",POLYLINE,...,-30.914061,-141.835938,-30.914061,-121.054688,-53.882809,-121.054688,22.968748,20.781250,-42.398435,-131.445312


In [40]:
# Let's slightly modify the table
# We need the actual referenced SeriesInstanceUID
# And we need the Dimensions, Pixel spacing IPP, especially IPP[2] for the z value of the bounding box

referenced_sop_instance_uid_list = list(df_sr['ReferencedSOPInstanceUID'].values)

client_bq = bigquery.Client(project=project_name)

query = f"""
    SELECT
      PatientID,
      StudyInstanceUID,
      SeriesInstanceUID,
      SOPInstanceUID,
      `Rows` as num_rows,
      `Columns` as num_columns,
      PixelSpacing,
      ImagePositionPatient
    FROM
      `bigquery-public-data.idc_current.dicom_all`
    WHERE
      SOPInstanceUID IN UNNEST(@referenced_sop_instance_uid_list)
    ORDER BY
      PatientID,
      StudyInstanceUID,
      SeriesInstanceUID,
      ImagePositionPatient[SAFE_OFFSET(2)]
      """

job_config = bigquery.QueryJobConfig(query_parameters=[bigquery.ArrayQueryParameter("referenced_sop_instance_uid_list", "STRING", referenced_sop_instance_uid_list)])
df_idc = client_bq.query(query, job_config=job_config).to_dataframe()

In [41]:
# Reformat the PixelSpacing and the ImagePositionPatient columns

df_idc['pixel_spacing_x'] = [np.float32(f[0]) for f in df_idc['PixelSpacing'].values]
df_idc['pixel_spacing_y'] = [np.float32(f[1]) for f in df_idc['PixelSpacing'].values]
df_idc['ipp0'] = [np.float32(f[0]) for f in df_idc['ImagePositionPatient'].values]
df_idc['ipp1'] = [np.float32(f[1]) for f in df_idc['ImagePositionPatient'].values]
df_idc['ipp2'] = [np.float32(f[2]) for f in df_idc['ImagePositionPatient'].values]

df_idc = df_idc[['PatientID', 'StudyInstanceUID', 'SeriesInstanceUID', 'SOPInstanceUID',
                 'num_rows', 'num_columns',
                 'pixel_spacing_x', 'pixel_spacing_y',
                 'ipp0', 'ipp1', 'ipp2']]

In [42]:
df_idc.head()

,PatientID,StudyInstanceUID,SeriesInstanceUID,SOPInstanceUID,num_rows,num_columns,pixel_spacing_x,pixel_spacing_y,ipp0,ipp1,ipp2
0,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.29991037322734048580038819...,512,512,0.585938,0.585938,-149.707031,-319.707031,-76.400002
1,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.27443508115501826384206327...,512,512,0.585938,0.585938,-149.707031,-319.707031,-78.400002
2,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.16899951679153198601142574...,512,512,0.585938,0.585938,-149.707031,-319.707031,-80.400002
3,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.17481124987277919843449170...,512,512,0.585938,0.585938,-149.707031,-319.707031,-82.400002
4,100012,1.2.840.113654.2.55.23803494144550801138646327...,1.2.840.113654.2.55.24023112856488152536348979...,1.2.840.113654.2.55.29574962348509387538142601...,512,512,0.585938,0.585938,-149.707031,-319.707031,-84.400002


## df_nlst_metadata - get the associated clinical metadata - for classification

In [44]:
# Rewrite the query below but faster
client = bigquery.Client(project=project_name, location='US') # since below can't mix US and us-central1
df_needed = df_idc[['PatientID', 'SeriesInstanceUID', 'SOPInstanceUID']].drop_duplicates()
table_id = "idc-external-018.nlst_sybil_fm_demo.needed_uids"
job_config = bigquery.LoadJobConfig(
        write_disposition=bigquery.job.WriteDisposition.WRITE_TRUNCATE
    )
client.load_table_from_dataframe(df_needed, table_id, job_config=job_config).result()

LoadJob<project=idc-external-018, location=US, id=82fce856-e50d-48a8-bb4a-e0e92524df7f>

In [45]:
# Here we get the staging data

query = """
WITH dicom_mapped AS (
  SELECT
    PatientID,
    StudyInstanceUID,
    StudyDate,
    SeriesInstanceUID,
    SOPInstanceUID,
    InstanceNumber,
    `Rows`,
    `Columns`,
    -- Mapping StudyDate to numerical values
    CASE StudyDate
      WHEN '1999-01-02' THEN 0
      WHEN '2000-01-02' THEN 1
      WHEN '2001-01-02' THEN 2
      ELSE 3
    END AS StudyDate_mapped,
    COUNT(*) OVER (PARTITION BY SeriesInstanceUID) AS sop_count_per_series
  FROM `bigquery-public-data.idc_current.dicom_all`
  WHERE SeriesInstanceUID IN (
    SELECT DISTINCT SeriesInstanceUID
    FROM `idc-external-018.nlst_sybil_fm_demo.needed_uids`
  )
)

SELECT
  ctab.dicom_patient_id AS PatientID,
  dicom_mapped.StudyInstanceUID,
  dicom_mapped.StudyDate,
  dicom_mapped.SeriesInstanceUID,
  dicom_mapped.SOPInstanceUID,
  ctab.sct_slice_num,
  ctab.study_yr,
  dicom_mapped.Rows,
  dicom_mapped.Columns,
  dicom_mapped.sop_count_per_series,
  -- Map de_stag to de_stag_mapped using your dictionary
  CASE prsn.de_stag
    WHEN '110' THEN 0
    WHEN '120' THEN 1
    WHEN '210' THEN 2
    WHEN '220' THEN 3
    WHEN '310' THEN 4
    WHEN '320' THEN 5
    WHEN '400' THEN 6
    ELSE -1
  END AS de_stag_mapped
FROM
  `bigquery-public-data.idc_current_clinical.nlst_ctab` AS ctab
JOIN
  `bigquery-public-data.idc_current_clinical.nlst_prsn` AS prsn
  ON prsn.dicom_patient_id = ctab.dicom_patient_id
JOIN
  dicom_mapped
  ON dicom_mapped.InstanceNumber = ctab.sct_slice_num
  AND ctab.study_yr = dicom_mapped.StudyDate_mapped
JOIN
  `idc-external-018.nlst_sybil_fm_demo.needed_uids` AS needed
  ON needed.PatientID = prsn.dicom_patient_id
  AND needed.SeriesInstanceUID = dicom_mapped.SeriesInstanceUID
  AND needed.SOPInstanceUID = dicom_mapped.SOPInstanceUID
WHERE
  -- Only keep rows where de_stag_mapped is not -1
  CASE prsn.de_stag
    WHEN '110' THEN 0 # "Stage IA"
    WHEN '120' THEN 1 # "Stage IB"
    WHEN '210' THEN 2 # "Stage IIA"
    WHEN '220' THEN 3 # "Stage IIB"
    WHEN '310' THEN 4 # "Stage IIIA"
    WHEN '320' THEN 5 # "Stage IIIB"
    WHEN '400' THEN 6 # "Stage IV"
    ELSE -1
  END != -1
"""
df_nlst_metadata = client_bq.query(query).to_dataframe()

In [46]:
df_nlst_metadata.head()

,PatientID,StudyInstanceUID,StudyDate,SeriesInstanceUID,SOPInstanceUID,sct_slice_num,study_yr,Rows,Columns,sop_count_per_series,de_stag_mapped
0,130975,1.2.840.113654.2.55.20305085858975232018329927...,2000-01-02,1.2.840.113654.2.55.12881149774656140035795514...,1.2.840.113654.2.55.10995924436238550073736484...,39,1,512,512,120,4
1,129695,1.2.840.113654.2.55.31352591972969926371652665...,2001-01-02,1.2.840.113654.2.55.15984899811178273278176379...,1.2.840.113654.2.55.64838499121538104643406339...,105,2,512,512,130,1
2,129695,1.2.840.113654.2.55.31352591972969926371652665...,2001-01-02,1.2.840.113654.2.55.15984899811178273278176379...,1.2.840.113654.2.55.26318475184610510764140564...,77,2,512,512,130,1
3,106204,1.2.840.113654.2.55.12577199462825813436791158...,2000-01-02,1.2.840.113654.2.55.31822289743106374456653341...,1.2.840.113654.2.55.11315356465995879472001339...,57,1,512,512,160,0
4,109573,1.2.840.113654.2.55.12048244682320285659632520...,1999-01-02,1.2.840.113654.2.55.32965871416156269882142496...,1.2.840.113654.2.55.16849491099337711024720925...,91,0,512,512,122,0


## Join the tables to hold the SR info and clinical metadata info

In [82]:
# Then join with df_idc
df_sr_join = df_sr.merge(df_idc,
                         left_on=['ReferencedSOPInstanceUID'],
                         right_on=['SOPInstanceUID'],
                         suffixes=('','_right'))
# Drop the duplicate column from the right dataframe
df_sr_join = df_sr_join.drop(columns=['PatientID_right','SOPInstanceUID_right'])

# Then join with the df_nlst_metadata
df_sr_and_nlst = df_sr_join.merge(df_nlst_metadata,
                                  left_on=['ReferencedSOPInstanceUID'],
                                  right_on=['SOPInstanceUID'],
                                  suffixes=('','_right'))
df_sr_and_nlst = df_sr_and_nlst.drop(columns=['PatientID_right','StudyInstanceUID_right','SeriesInstanceUID_right','num_rows', 'num_columns'])
# Rename columns
df_sr_and_nlst = df_sr_and_nlst.rename({'trackingIdentifier': 'TrackingIdentifier',
                                        'trackingUniqueIdentifier':'TrackingUID',
                                        'finding':'FindingType',
                                        'findingSite': 'FindingSite',
                                        'ReferencedSOPInstanceUID':'SOPInstanceUID'}, axis=1)
# Reorder the columns
df_sr_and_nlst = df_sr_and_nlst[['PatientID', 'StudyInstanceUID', 'StudyDate', 'study_yr', 'SeriesInstanceUID', 'SR_SeriesInstanceUID', 'sop_count_per_series',
                                 'TrackingIdentifier', 'TrackingUID', 'SOPInstanceUID',
                                 'FindingType', 'FindingSite',
                                 'pixel_spacing_x', 'pixel_spacing_y',
                                 'width', 'height', 'center_x', 'center_y', 'ipp0', 'ipp1', 'ipp2',
                                 'sct_slice_num', 'de_stag_mapped']]
# Order the values in the columns
df_sr_and_nlst = df_sr_and_nlst.sort_values(by=['PatientID', 'study_yr', 'StudyInstanceUID', 'SeriesInstanceUID', 'TrackingIdentifier'])
df_sr_and_nlst.head()



,PatientID,StudyInstanceUID,StudyDate,study_yr,SeriesInstanceUID,SR_SeriesInstanceUID,sop_count_per_series,TrackingIdentifier,TrackingUID,SOPInstanceUID,...,pixel_spacing_y,width,height,center_x,center_y,ipp0,ipp1,ipp2,sct_slice_num,de_stag_mapped
1,100012,1.2.840.113654.2.55.23803494144550801138646327...,1999-01-02,0,1.2.840.113654.2.55.24023112856488152536348979...,1.2.826.0.1.3680043.8.498.85406351932512765678...,162,4,1.2.826.0.1.3680043.8.498.77711953470893226641...,1.2.826.0.1.3680043.8.498.65416915969156279384...,...,0.585938,24.023439,21.679688,-32.812499,-137.773438,-149.707031,-319.707031,-92.400002,38,0
0,100012,1.2.840.113654.2.55.38321092839390108338558865...,2000-01-02,1,1.2.840.113654.2.55.50761756412482430061802871...,1.2.826.0.1.3680043.8.498.37369100471267400346...,157,6,1.2.826.0.1.3680043.8.498.53715998280437478378...,1.2.826.0.1.3680043.8.498.47967329963589463487...,...,0.546875,22.968748,21.328125,-42.945310,-130.078125,-133.726562,-309.726562,1220.699951,39,0
2,100147,1.2.840.113654.2.55.13303292650860633016545772...,1999-01-02,0,1.2.840.113654.2.55.24785488463405747713776937...,1.2.826.0.1.3680043.8.498.22958024421071113143...,110,3,1.2.826.0.1.3680043.8.498.80000752432193002275...,1.2.826.0.1.3680043.8.498.19096797202406945100...,...,0.660156,25.746078,24.425772,-77.946526,37.958905,-177.300003,-169.000000,-51.189999,88,0
3,100147,1.2.840.113654.2.55.31958452963320032523273261...,2000-01-02,1,1.2.840.113654.2.55.15708941008648745210499888...,1.2.826.0.1.3680043.8.498.37167572880559873170...,116,5,1.2.826.0.1.3680043.8.498.19272968908902719452...,1.2.826.0.1.3680043.8.498.35090590045333262510...,...,0.644531,23.847645,23.847647,-69.931677,25.406564,-165.000000,-188.899994,-49.275002,92,0
4,100158,1.2.840.113654.2.55.81185422866512279860334872...,2001-01-02,2,1.2.840.113654.2.55.31060976780967844152296392...,1.2.826.0.1.3680043.8.498.46017404573084441864...,146,4,1.2.826.0.1.3680043.8.498.24685678672202097011...,1.2.826.0.1.3680043.8.498.12620021782755699377...,...,0.683594,18.457039,19.140631,-108.348026,10.937568,-179.100006,-175.000000,-149.440002,57,0


In [83]:
# Let's get the counts of the de_stag_mapped
df_sr_and_nlst_counts = df_sr_and_nlst['de_stag_mapped'].value_counts().sort_index()
df_sr_and_nlst_counts

,count
de_stag_mapped,
0,424
1,115
2,21
3,19
4,64
5,48
6,87


# Temporarily save out csv file to Google Drive

In [84]:
df_sr_and_nlst.to_csv("/content/nlst_sybil_fm.csv")

In [85]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [86]:
!cp "/content/nlst_sybil_fm.csv" "/content/gdrive/MyDrive/Colab Notebooks/SR_NLST_Sybil/demo/"